## Generate multi-view images with Zero123++

Given a single reference image, generate consistent multi-view renders using the pretrained [Zero123++](https://github.com/SUDO-AI-3D/zero123plus) diffusion model.

In [ ]:
from huggingface_hub import snapshot_download

# Optional: pre-fetch the model weights before running the pipeline below.
snapshot_download(repo_id="sudo-ai/zero123plus-v1.1")


In [ ]:
import torch
import requests
from PIL import Image
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

# Install missing dependency first
# !pip install accelerate  # Uncomment if not installed

# Load pipeline from local module
pipeline = DiffusionPipeline.from_pretrained(
    "sudo-ai/zero123plus-v1.1",
    custom_pipeline="zero123plus_pipeline.pipeline",  # Use the local module path
    torch_dtype=torch.float16
)

# Configure scheduler
pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(
    pipeline.scheduler.config, timestep_spacing='trailing'
)
pipeline.to('cuda:0')

# Load input image
cond = Image.open(requests.get("https://d.skis.ltd/nrp/sample-data/lysol.png", stream=True).raw)

# Generate output
result = pipeline(cond, num_inference_steps=75).images[0]
result.save("output.png")